# Creative Advertising Multi-Agent System
## AI Agentic Bootcamp — Part B

This notebook implements a **sequential multi-agent advertising pipeline** using the **OpenAI Agents SDK**.

### Agent Pipeline
```
User Prompt
    ↓
Creative Director Agent  →  defines campaign strategy & brief
    ↓
Strategist Agent         →  develops target audience & messaging pillars
    ↓
Copywriter Agent         →  produces final ad copy & taglines
```

### Test Prompt
> *"Launch a campaign for a new eco-friendly water bottle in Bali"*

## 1. Setup & Imports

In [23]:
# Install the OpenAI Agents SDK (run once)
# !pip install openai-agents

import os
import asyncio
from agents import Agent, Runner, handoff

print("OpenAI Agents SDK imported successfully.")

OpenAI Agents SDK imported successfully.


## 2. Configure API Key

In [26]:
# Set your OpenAI API key
# Option A: load from environment variable (recommended)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Option B: load from .env file
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Please set it before running the pipeline.")
else:
    print(f"API key loaded: {api_key[:8]}...")

API key loaded: sk-proj-...


## 3. Define the Three Agents

Each agent has a focused role and passes its output to the next agent in the chain.

In [27]:
# ─────────────────────────────────────────────────────────────
# Agent 1: Creative Director
# Role: Sets the overall creative vision and campaign brief
# ─────────────────────────────────────────────────────────────
creative_director = Agent(
    name="Creative Director",
    model="gpt-4o-mini",
    instructions=(
        "You are a senior Creative Director at a top advertising agency. "
        "Your job is to receive a campaign request and produce a concise "
        "creative brief that includes:\n"
        "1. Campaign name and tagline concept\n"
        "2. Core brand message (1-2 sentences)\n"
        "3. Visual/tonal direction (mood, aesthetic, style)\n"
        "4. Key campaign objectives (brand awareness, conversions, etc.)\n\n"
        "Be bold, creative, and culturally aware. Your brief will be handed off "
        "to a Strategist to develop further. Keep output under 300 words."
    ),
)

print("Creative Director agent defined.")

Creative Director agent defined.


In [18]:
# ─────────────────────────────────────────────────────────────
# Agent 2: Strategist
# Role: Turns the creative brief into a strategic plan
# ─────────────────────────────────────────────────────────────
strategist = Agent(
    name="Strategist",
    model="gpt-4o-mini",
    instructions=(
        "You are a Brand Strategist specializing in digital and experiential marketing. "
        "You receive a creative brief and develop a strategic plan covering:\n"
        "1. Target audience profile (demographics, psychographics, behaviors)\n"
        "2. Three messaging pillars (core themes that resonate with the audience)\n"
        "3. Recommended channels (social media, influencers, events, etc.)\n"
        "4. Key campaign hooks or angles to stand out in the market\n\n"
        "Be data-informed and audience-centric. Your strategy will be handed off "
        "to a Copywriter. Keep output under 350 words."
    ),
)

print("Strategist agent defined.")

Strategist agent defined.


In [28]:
# ─────────────────────────────────────────────────────────────
# Agent 3: Copywriter
# Role: Produces the final ad copy from the strategy
# ─────────────────────────────────────────────────────────────
copywriter = Agent(
    name="Copywriter",
    model="gpt-4o-mini",
    instructions=(
        "You are an award-winning Copywriter known for crafting memorable, "
        "conversion-driven ad copy. You receive a creative brief and strategy, "
        "then produce:\n"
        "1. Hero tagline (short, punchy, memorable)\n"
        "2. Instagram caption (150 chars max, with 5 hashtags)\n"
        "3. Short-form video script (15-second hook + call-to-action)\n"
        "4. Email subject line + preview text\n"
        "5. One print ad headline with supporting body copy (50 words)\n\n"
        "Tailor all copy to the target audience and cultural context. "
        "Make every word earn its place."
    ),
)

print("Copywriter agent defined.")

Copywriter agent defined.


## 4. Build the Sequential Pipeline

The pipeline passes context forward: each agent sees the accumulated output of all previous agents.

In [29]:
async def run_advertising_pipeline(campaign_prompt: str) -> dict:
    """
    Run the sequential Creative Director → Strategist → Copywriter pipeline.
    
    Args:
        campaign_prompt: The initial campaign request from the user.
    
    Returns:
        Dictionary with outputs from each agent.
    """
    results = {}
    
    print("=" * 60)
    print("CREATIVE ADVERTISING MULTI-AGENT PIPELINE")
    print("=" * 60)
    print(f"Campaign Request: {campaign_prompt}\n")
    
    # ── Step 1: Creative Director ──────────────────────────────
    print("[1/3] Creative Director is crafting the brief...")
    director_result = await Runner.run(
        creative_director,
        input=f"Campaign request: {campaign_prompt}"
    )
    results["creative_brief"] = director_result.final_output
    print("\n📋 CREATIVE BRIEF:")
    print("-" * 40)
    print(director_result.final_output)
    
    # ── Step 2: Strategist ─────────────────────────────────────
    print("\n[2/3] Strategist is developing the strategy...")
    strategy_input = (
        f"Original campaign request: {campaign_prompt}\n\n"
        f"Creative Brief from Creative Director:\n{director_result.final_output}"
    )
    strategist_result = await Runner.run(
        strategist,
        input=strategy_input
    )
    results["strategy"] = strategist_result.final_output
    print("\n🎯 STRATEGY:")
    print("-" * 40)
    print(strategist_result.final_output)
    
    # ── Step 3: Copywriter ─────────────────────────────────────
    print("\n[3/3] Copywriter is producing the ad copy...")
    copy_input = (
        f"Original campaign request: {campaign_prompt}\n\n"
        f"Creative Brief:\n{director_result.final_output}\n\n"
        f"Strategy:\n{strategist_result.final_output}"
    )
    copywriter_result = await Runner.run(
        copywriter,
        input=copy_input
    )
    results["copy"] = copywriter_result.final_output
    print("\n✍️  FINAL AD COPY:")
    print("-" * 40)
    print(copywriter_result.final_output)
    
    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print("=" * 60)
    
    return results

print("Pipeline function defined.")

Pipeline function defined.


## 5. Run the Pipeline — Test Prompt

> **Test Prompt**: *"Launch a campaign for a new eco-friendly water bottle in Bali"*

In [30]:
# Define the test prompt from the rubric
TEST_PROMPT = "Launch a campaign for a new eco-friendly water bottle in Bali"

# Run the pipeline (works in both Jupyter and standard Python)
results = await run_advertising_pipeline(TEST_PROMPT)

CREATIVE ADVERTISING MULTI-AGENT PIPELINE
Campaign Request: Launch a campaign for a new eco-friendly water bottle in Bali

[1/3] Creative Director is crafting the brief...


OPENAI_API_KEY is not set, skipping trace export



📋 CREATIVE BRIEF:
----------------------------------------
**Campaign Name:** “Breathe Easy, Drink Clean”  
**Tagline Concept:** “Sip Sustainably, Live Responsibly”

**Core Brand Message:**  
Our eco-friendly water bottle is crafted for the conscious traveler, merging sustainable design with the beauty of Bali. With every sip, you’re not just quenching your thirst; you’re nurturing our planet.

**Visual/Tonal Direction:**  
The campaign will immerse viewers in the stunning landscapes of Bali, emphasizing its lush greenery and pristine beaches. The mood will be vibrant and uplifting, characterized by warm earthy tones and fluid, organic shapes. The aesthetic will combine natural textures with sleek, modern design, showcasing the bottle’s eco-friendly materials. Imagery will feature diverse locals and travelers in serene settings, highlighting the harmony between lifestyle and environmental stewardship. The tone will be friendly, aspirational, yet grounded in authenticity.

**Key Campai

OPENAI_API_KEY is not set, skipping trace export



🎯 STRATEGY:
----------------------------------------
**Strategic Plan for “Breathe Easy, Drink Clean” Campaign**

**1. Target Audience Profile:**  
- **Demographics:**  
  - Age: 18-35  
  - Gender: All  
  - Income: Middle to upper-middle class  
  - Location: Tourists and locals in Bali, Indonesia  
- **Psychographics:**  
  - Values sustainability and eco-friendliness  
  - Seeks adventure and unique experiences  
  - Prioritizes health and wellness  
  - Engaged in social and environmental causes  
- **Behaviors:**  
  - Actively participates in social media, especially Instagram and TikTok  
  - Frequently shares travel experiences online  
  - Engages with brands that align with their values  

**2. Three Messaging Pillars:**  
- **Sustainability:** Highlight the eco-friendly materials and production processes that reduce environmental impact.  
- **Connection to Nature:** Foster a connection between mindful living and the natural beauty of Bali, emphasizing the importance of pr

OPENAI_API_KEY is not set, skipping trace export



✍️  FINAL AD COPY:
----------------------------------------
### 1. Hero Tagline
**“Sip Sustainably, Live Responsibly.”**

### 2. Instagram Caption
“Transform your hydration game with our eco-friendly bottle! 🌿💧 #SipSustainably #BaliLifestyle #EcoFriendly #AdventureAwaits #DrinkClean"

### 3. Short-form Video Script
**[Scene: A breathtaking sunrise over Bali’s beaches, the bottle glinting in the light.]**  
**Narrator:** “In Bali, every sip counts.”  
**[Cut to diverse travelers enjoying water from the bottle while hiking and relaxing.]**  
**Narrator:** “Join the movement for a cleaner planet.”  
**[Scene shifts to a beach clean-up in action.]**  
**Narrator:** “Choose sustainability. Choose us.”  
**[Text on screen: “Drink Clean, Live Green. Tap the link!”]**

### 4. Email Subject Line + Preview Text
**Subject Line:** “Sip Sustainably & Explore Bali’s Beauty! 🌍💚”  
**Preview Text:** “Discover the eco-friendly water bottle shaping mindful adventures. Drink clean and join our community

OPENAI_API_KEY is not set, skipping trace export


## 6. Save Output to File

In [32]:
# Save the full pipeline output to a text file for submission
output_path = "sample_output.txt"

with open(output_path, "w") as f:
    f.write("CREATIVE ADVERTISING MULTI-AGENT PIPELINE — SAMPLE OUTPUT\n")
    f.write("=" * 60 + "\n")
    f.write(f"Campaign Request: {TEST_PROMPT}\n\n")
    
    f.write("─" * 60 + "\n")
    f.write("[AGENT 1] CREATIVE DIRECTOR — Creative Brief\n")
    f.write("─" * 60 + "\n")
    f.write(results.get("creative_brief", "") + "\n\n")
    
    f.write("─" * 60 + "\n")
    f.write("[AGENT 2] STRATEGIST — Campaign Strategy\n")
    f.write("─" * 60 + "\n")
    f.write(results.get("strategy", "") + "\n\n")
    
    f.write("─" * 60 + "\n")
    f.write("[AGENT 3] COPYWRITER — Final Ad Copy\n")
    f.write("─" * 60 + "\n")
    f.write(results.get("copy", "") + "\n")

print(f"Output saved to: {output_path}")

Output saved to: sample_output.txt


## 7. How to Extend the Pipeline

You can add more agents (e.g., a **Media Planner** or **Brand Guardian**) by:
1. Defining a new `Agent(...)` with custom instructions
2. Running it with the accumulated context as input
3. Adding its output to the `results` dict

The sequential pattern ensures each agent builds on all prior work.